# core

> HTMX v4 support for FastHTML

In [ ]:
#| export
#| default_exp core

In [ ]:
#| export
import json

from fastcore.basics import patch
from fastcore.utils import *
from fastcore.xml import *
from fastcore.meta import delegates

from fasthtml.common import *
from fasthtml.starlette import *
from fasthtml.core import *

from fasthtml.core import _wrap_ex, _list, _get_htmx, _fix_anno, _wrap_ws, _params, _handle, _ws_endp, _find_ps, _is_body, _from_body, snake2hyphens, ApiReturn
from fasthtml.fastapp import _get_tbl, _app_factory

In [ ]:
#| export
from inspect import Parameter
empty = Parameter.empty

import fasthtml.core as _core
from fasthtml.core import _send_ws

from typing import Optional, get_type_hints, get_args, get_origin, Union, Mapping, TypedDict, List, Any
from types import UnionType, SimpleNamespace as ns, GenericAlias
from datetime import datetime,date

In [ ]:
from fasthtml.jupyter import *

In [ ]:
#| export
@delegates(ft_hx)
def Partial(*args, **kwargs): return ft_hx("hx-partial")(*args, **kwargs)


### `Partial` and OOB swap ordering

`Partial` creates an `<hx-partial>` tag, which is the htmx v4 recommended way to update multiple targets from a single response.

**htmx v4 guidance:**
- Use `hx_swap_oob=True` (plain ID matching) for simple same-element replacement
- Use `Partial(content, hx_target='#id', hx_swap='...')` everywhere else — it is more explicit and readable than selector-based OOB

**Swap ordering differences between v2 and v4:**

| Version | Order |
|---------|-------|
| htmx v2 | OOB swaps → primary swap |
| htmx v4 | Primary swap → OOB swaps → partials |

This ordering difference can cause subtle changes in DOM structure when migrating

# htmx4 hdrs and metaCharacter

In [ ]:
#| export
htmx4src   = Script(src="https://unpkg.com/htmx.org@4.0.0-alpha7/dist/htmx.js")

In [ ]:
#| export
# When htmx4=True, configures htmx v4 with metaCharacter="-"
def def_hdrs(htmx=True, htmx4=False, surreal=True):
    "Default headers for a FastHTML app"
    hdrs = []
    if surreal: hdrs = [surrsrc,scopesrc] + hdrs
    if htmx and htmx4: raise ValueError("Cannot enable both htmx and htmx4")
    if htmx: hdrs = [htmxsrc,fhjsscr] + hdrs
    if htmx4: 
        # metaCharacter="-" makes htmx4 use dashes instead of colons (Python-friendly)
        meta_cfg = Meta(name="htmx:config", content=json.dumps({"metaCharacter": "-"}))
        hdrs = [meta_cfg, htmx4src,fhjsscr] + hdrs 
    # TODO: Check if fhjsscr works with htmx4
    return [charset, viewport] + hdrs

# FastHTML and fastapp

In [ ]:
#| export
# Patch FastHTML.__init__ to add htmx4 support
# - Adds `htmx4` parameter to toggle htmx v4 headers
# - Add self.htmx4
# - Passes htmx4 to def_hdrs() which handles the header selection
# - Maps 'ws' and 'ws4' extensions to 'ws4' when htmx4=True
@patch
def __init__(self: FastHTML, debug=False, routes=None, middleware=None, title: str = "FastHTML page", exception_handlers=None,
                on_startup=None, on_shutdown=None, lifespan=None, hdrs=None, ftrs=None, exts=None,
                before=None, after=None, surreal=True, htmx=True, htmx4=False, default_hdrs=True, sess_cls=SessionMiddleware,
                secret_key=None, session_cookie='session_', max_age=365*24*3600, sess_path='/',
                same_site='lax', sess_https_only=False, sess_domain=None, key_fname='.sesskey',
                body_wrap=noop_body, htmlkw=None, nb_hdrs=False, canonical=True, **bodykw):
    middleware,before,after = map(_list, (middleware,before,after))
    self.title,self.canonical,self.session_cookie,self.key_fname = title,canonical,session_cookie,key_fname
    self.htmx4 = htmx4
    hdrs,ftrs,exts = map(listify, (hdrs,ftrs,exts))
    if htmx4 and exts:
        exts = ['ws4' if e in ('ws', 'ws4') else e for e in exts]
    exts = {k:htmx_exts[k] for k in exts}
    htmlkw = htmlkw or {}
    if default_hdrs: hdrs = def_hdrs(htmx, htmx4, surreal=surreal) + hdrs
    hdrs += [Script(src=ext) for ext in exts.values()]
    if IN_NOTEBOOK:
        hdrs.append(iframe_scr) # TODO: check iframe_scr if work with htmx4
        from IPython.display import display,HTML
        if nb_hdrs: display(HTML(to_xml(tuple(hdrs))))
        middleware.append(cors_allow)
    on_startup,on_shutdown = listify(on_startup) or None,listify(on_shutdown) or None
    self.lifespan,self.hdrs,self.ftrs = lifespan,hdrs,ftrs
    self.body_wrap,self.before,self.after,self.htmlkw,self.bodykw = body_wrap,before,after,htmlkw,bodykw
    self.secret_key = get_key(secret_key, key_fname)
    if sess_cls:
        sess = Middleware(sess_cls, secret_key=self.secret_key,session_cookie=session_cookie,
                            max_age=max_age, path=sess_path, same_site=same_site,
                            https_only=sess_https_only, domain=sess_domain)
        middleware.append(sess)
    exception_handlers = ifnone(exception_handlers, {})
    if 404 not in exception_handlers:
        def _not_found(req, exc): return  Response('404 Not Found', status_code=404)
        exception_handlers[404] = _not_found
    excs = {k:_wrap_ex(v, k, hdrs, ftrs, htmlkw, bodykw, body_wrap=body_wrap) for k,v in exception_handlers.items()}
    super(FastHTML, self).__init__(debug, routes, middleware=middleware, exception_handlers=excs, on_startup=on_startup, on_shutdown=on_shutdown, lifespan=lifespan)

In [ ]:
#| export
# Supports htmx4=True for htmx v4 compatibility
def fast_app(
        db_file:Optional[str]=None, # Database file name, if needed
        render:Optional[callable]=None, # Function used to render default database class
        hdrs:Optional[tuple]=None, # Additional FT elements to add to <HEAD>
        ftrs:Optional[tuple]=None, # Additional FT elements to add to end of <BODY>
        tbls:Optional[dict]=None, # Experimental mapping from DB table names to dict table definitions
        before:Optional[tuple]|Beforeware=None, # Functions to call prior to calling handler
        middleware:Optional[tuple]=None, # Standard Starlette middleware
        live:bool=False, # Enable live reloading
        debug:bool=False, # Passed to Starlette, indicating if debug tracebacks should be returned on errors
        title:str="FastHTML page", # Default page title
        routes:Optional[tuple]=None, # Passed to Starlette
        exception_handlers:Optional[dict]=None, # Passed to Starlette
        on_startup:Optional[callable]=None, # Passed to Starlette
        on_shutdown:Optional[callable]=None, # Passed to Starlette
        lifespan:Optional[callable]=None, # Passed to Starlette
        default_hdrs=True, # Include default FastHTML headers such as HTMX script?
        pico:Optional[bool]=None, # Include PicoCSS header?
        surreal:Optional[bool]=True, # Include surreal.js/scope headers?
        htmx:Optional[bool]=True, # Include HTMX header?
        htmx4:Optional[bool]=False, # Include HTMX4 header?
        exts:Optional[list|str]=None, # HTMX extension names to include
        canonical:bool=True, # Automatically include canonical link?
        secret_key:Optional[str]=None, # Signing key for sessions
        key_fname:str='.sesskey', # Session cookie signing key file name
        session_cookie:str='session_', # Session cookie name
        max_age:int=365*24*3600, # Session cookie expiry time
        sess_path:str='/', # Session cookie path
        same_site:str='lax', # Session cookie same site policy
        sess_https_only:bool=False, # Session cookie HTTPS only?
        sess_domain:Optional[str]=None, # Session cookie domain
        htmlkw:Optional[dict]=None, # Attrs to add to the HTML tag
        bodykw:Optional[dict]=None, # Attrs to add to the Body tag
        reload_attempts:Optional[int]=1, # Number of reload attempts when live reloading
        reload_interval:Optional[int]=1000, # Time between reload attempts in ms
        static_path:str=".",  # Where the static file route points to, defaults to root dir
        body_wrap:callable=noop_body, # FT wrapper for body contents
        nb_hdrs:bool=False, # If in notebook include headers inject headers in notebook DOM?
        **kwargs):
    "Create a FastHTML or FastHTMLWithLiveReload app."
    h = (picolink,) if pico or (pico is None and default_hdrs) else ()
    if hdrs: h += tuple(hdrs)

    app = _app_factory(hdrs=h, ftrs=ftrs, before=before, middleware=middleware, live=live, debug=debug, title=title, routes=routes, exception_handlers=exception_handlers,
                  on_startup=on_startup, on_shutdown=on_shutdown, lifespan=lifespan, default_hdrs=default_hdrs, secret_key=secret_key, canonical=canonical,
                  session_cookie=session_cookie, max_age=max_age, sess_path=sess_path, same_site=same_site, sess_https_only=sess_https_only,
                  sess_domain=sess_domain, key_fname=key_fname, exts=exts, surreal=surreal, htmx=htmx, htmx4=htmx4, htmlkw=htmlkw,
                  reload_attempts=reload_attempts, reload_interval=reload_interval, body_wrap=body_wrap, nb_hdrs=nb_hdrs, **(bodykw or {}))
    app.static_route_exts(static_path=static_path)
    if not db_file: return app,app.route

    db = database(db_file)
    if not tbls: tbls={}
    if kwargs:
        if isinstance(first(kwargs.values()), dict): tbls = kwargs
        else:
            kwargs['render'] = render
            tbls['items'] = kwargs
    dbtbls = [_get_tbl(db.t, k, v) for k,v in tbls.items()]
    if len(dbtbls)==1: dbtbls=dbtbls[0]
    return app,app.route,*dbtbls

# Request Headers

## Migrating request headers from htmx v2 to v4

**Key Differences:**

| v2 Header | v4 Header | v2 Value | v4 Value |
|-----------|-----------|----------|----------|
| `HX-Trigger` | `HX-Source` | Element id (e.g. `textid`) | `tag#id` (e.g. `p#textid`) |
| `HX-Trigger-Name` | *(removed)* | Element name | — |
| `HX-Target` | `HX-Target` | Element id (e.g. `dest`) | `tag#id` (e.g. `p#dest`) |
| *(n/a)* | `HX-Request-Type` | — | `"partial"` or `"full"` |
| `HX-Prompt` | *(removed)* | User input | — |

**Impact on FastHTML handlers:**
- `hx_trigger: str` → `hx_source: str`
- The value format changed: v2 sends just the id, v4 sends `tag#id`
- `HtmxHeaders` dataclass may need updating to reflect new header names

**Note:** `hx-trigger` the *HTML attribute* still works the same in v4 — only the *request header* changed.

## Summary

Patches needed to make request headers work with htmx v4:

### 1. **`htmx_hdrs` dict**
- Added v4-only header mappings: `request_type` → `HX-Request-Type`, `source` → `HX-Source`
- These are added to the existing upstream `_core.htmx_hdrs` dict so `_get_htmx` can find them

### 2. **`HtmxHeaders` dataclass**
- Extended with two new v4 fields: `request_type` and `source`
- Replaces the upstream dataclass via `_core.HtmxHeaders = HtmxHeaders`
- Existing v2 fields (`trigger`, `trigger_name`, `prompt`, etc.) are preserved for backward compatibility

In [ ]:
# Add v4-only headers to the existing upstream dict
_core.htmx_hdrs.update(request_type="HX-Request-Type", source="HX-Source")

In [ ]:
@dataclass
class HtmxHeaders:
    boosted:str|None=None; current_url:str|None=None; history_restore_request:str|None=None; prompt:str|None=None
    request:str|None=None; target:str|None=None; trigger_name:str|None=None; trigger:str|None=None
    request_type:str|None=None; source:str|None=None

_core.HtmxHeaders = HtmxHeaders


## Request Headers Example

### htmx v2 example

```python
app, rt = fast_app()

@rt
def index():
    return Div(
        P(id="textid", name="textname", hx_post=handle, hx_target='#dest')(
            "Click Me"),
        P("Text to Update", id="dest"))

@rt
async def handle(hx_trigger:str): return Div(f"trigger: {hx_trigger}")

srv = JupyUvi(app)
```

Expected output when clicking: `trigger: textid`

### htmx v4 example

```python
app, rt = fast_app(htmx=False, htmx4=True)

@rt
def index():
    return Div(
        P(id="textid", name="textname", hx_post=handle, hx_target='#dest')(
            "Click Me"),
        P("Text to Update", id="dest"))

@rt
async def handle(htmx: HtmxHeaders):
    return Div(f"source: {htmx.source}, target: {htmx.target}, request_type: {htmx.request_type}")

srv = JupyUvi(app)
```

Expected output when clicking: `source: p#textid, target: p#dest, request_type: partial`

**Notes:**
- In v4, use `hx_source: str` instead of `hx_trigger: str` for the raw header value
- Use `htmx: HtmxHeaders` to get all headers as a structured object
- The `target` value in v4 is `tag#id` format (e.g. `p#dest`) instead of just the id (`dest`)

# Events

## Migrating Events from htmx v2 to v4

**Official v4 events docs:** https://four.htmx.org/events/

**Key Differences:**

| Change | htmx v2 | htmx v4 |
|--------|---------|---------|
| Event naming | camelCase (e.g. `htmx:beforeRequest`) | colon-separated (e.g. `htmx:before:request`) |
| Event detail | Flat properties (`event.detail.elt`) | Unified `ctx` object (`event.detail.ctx.sourceElement`) |

**Common event name mappings:**

| v2 | v4 |
|----|-----|
| `htmx:beforeRequest` | `htmx:before:request` |
| `htmx:afterRequest` | `htmx:after:request` |
| `htmx:beforeSwap` | `htmx:before:swap` |
| `htmx:afterSwap` | `htmx:after:swap` |

**Watch out for detail structure changes:** The event `detail` object was reorganised in v4 into a unified `ctx` object. Always check the v4 event docs for the correct property path. For example, `event.detail.elt` → `event.detail.ctx.sourceElement`, and `event.detail.path` → `event.detail.ctx.request.action`.

See the full event mapping in the CRAFT htmx v4 guide under "Event Name Mappings".

## PicoBusy4

`PicoBusy()` uses htmx events to show a Pico CSS loading spinner on the triggering element via `aria-busy`. In v4, both the event names and the detail structure changed:

In [ ]:
#| export
def PicoBusy4():
    return (HtmxOn('before:request', "event.detail.ctx.sourceElement.setAttribute('aria-busy', 'true' )"),
            HtmxOn('after:request',  "event.detail.ctx.sourceElement.setAttribute('aria-busy', 'false')"))

## sid_scr4

`sid_scr` uses htmx events to inject a session ID into every request URL. In v4, two things changed:

| Change | v2 | v4 |
|--------|----|----|
| Event name | `htmx:configRequest` | `htmx:config:request` |
| URL access | `e.detail.path` | `e.detail.ctx.request.action` |

In [ ]:
#| export
sid_scr4 = Script("""
function uuid() {
    return [...crypto.getRandomValues(new Uint8Array(10))].map(b=>b.toString(36)).join('');
}

sessionStorage.setItem("sid", sessionStorage.getItem("sid") || uuid());

htmx.on("htmx:config:request", (e) => {
    const sid = sessionStorage.getItem("sid");
    if (sid) {
        const url = new URL(e.detail.ctx.request.action, window.location.origin);
        url.searchParams.set('sid', sid);
        e.detail.ctx.request.action = url.pathname + url.search;
    }
});
""")

# WS

## Migrating WebSockets from htmx v2 to v4

**Key Differences:**

**htmx v2:**
- Requires WS extension: `Script(src="https://cdn.jsdelivr.net/npm/htmx-ext-ws@2.0.3/ws.js")`
- Uses `hx_ext="ws"`, `ws_connect="/endpoint"`, `ws_send=True`
- Server sends raw HTML directly: `await send(Div('Hello', id='msg'))`
- Client receives HTML and swaps based on `id` or `hx-swap-oob`

**htmx v4:**
- Requires WS extension: `Script(src="https://unpkg.com/htmx.org@4.0.0-alpha7/dist/ext/hx-ws.js")`
- Uses `hx_ws_connect="/endpoint"`, `hx_ws_send=True`
- Server sends raw HTML, same as v2 — uses `hx_swap_oob=True` or `<hx-partial>` for targeting
- No need for `hx_ext="ws"` attribute — extension auto-registers when loaded
- **Must explicitly set `hx_swap_oob=True`** on each element for ID-based swapping. In v2, elements with a matching `id` would swap implicitly; in v4 you must be explicit.

**Receiving form data (server-side):**
- **v2:** Form fields are at the **root** of the incoming JSON — e.g. `data['message']`
- **v4:** Form fields are nested under `values` — e.g. `data['values']['message']`

This is because v4 wraps everything in a JSON envelope with `type`, `request_id`, `headers`, `values`, etc., even when the *server* sends raw HTML back.

**Important:** Although htmx v4 WS supports a JSON envelope format for *server→client* messages too, we send raw HTML for compatibility with htmx v2. This means the same `_send_ws` function works for both versions. The main differences are the attribute names and the incoming data structure described above.

## Summary

Patches needed to make WS work with both htmx v2 and v4:

## 1. **Headers & Configuration**
- Created `htmx4src` pointing to htmx v4 alpha
- Modified `def_hdrs()` to add htmx v4 script + meta config with `metaCharacter="-"` (makes attributes use dashes instead of colons)
- Added `ws4` extension URL to `htmx_exts`

## 2. **FastHTML.__init__ Patch**
- Added `htmx4` parameter and stored it as `self.htmx4`
- Auto-maps `'ws'` extension to `'ws4'` when `htmx4=True`

## 3. **Request Headers** (see Request Headers section above)
- `HtmxHeaders` dataclass extended with v4 fields (`source`, `request_type`)
- `_get_htmx` picks `htmx2_hdrs` or `htmx4_hdrs` based on `htmx4` flag
- `_find_p` passes `htmx4` from `req.scope['app']` to `_get_htmx`

## 4. **WebSocket Header Extraction** (`_wrap_ws`)
- htmx v2 sends `HEADERS` (uppercase), v4 sends `headers` (lowercase)
- Patch checks both: `data.pop('HEADERS', {}) or data.pop('headers', {})`

## 5. **WebSocket Parameter Handling** (`_find_p`)
- htmx v4 nests form fields under `data['values']` instead of top-level `data`
- Patch adds: after `data.get(arg)`, also checks `data.get('values', {}).get(arg)` when `htmx4=True`
- Always uses `_send_ws` (raw HTML) for both v2 and v4 — no separate `_send_ws4` needed


In [ ]:
#| export
from inspect import Parameter
empty = Parameter.empty

import fasthtml.core as _core
from fasthtml.core import _send_ws

from typing import Optional, get_type_hints, get_args, get_origin, Union, Mapping, TypedDict, List, Any
from types import UnionType, SimpleNamespace as ns, GenericAlias
from datetime import datetime,date


In [ ]:
from starlette.testclient import TestClient

In [ ]:
htmx_exts

{'morph': 'https://cdn.jsdelivr.net/npm/idiomorph@0.7.3/dist/idiomorph-ext.min.js',
 'head-support': 'https://cdn.jsdelivr.net/npm/htmx-ext-head-support@2.0.4/head-support.js',
 'preload': 'https://cdn.jsdelivr.net/npm/htmx-ext-preload@2.1.1/preload.js',
 'class-tools': 'https://cdn.jsdelivr.net/npm/htmx-ext-class-tools@2.0.1/class-tools.js',
 'loading-states': 'https://cdn.jsdelivr.net/npm/htmx-ext-loading-states@2.0.1/loading-states.js',
 'multi-swap': 'https://cdn.jsdelivr.net/npm/htmx-ext-multi-swap@2.0.0/multi-swap.js',
 'path-deps': 'https://cdn.jsdelivr.net/npm/htmx-ext-path-deps@2.0.0/path-deps.js',
 'remove-me': 'https://cdn.jsdelivr.net/npm/htmx-ext-remove-me@2.0.0/remove-me.js',
 'debug': 'https://unpkg.com/htmx.org@1.9.12/dist/ext/debug.js',
 'ws': 'https://cdn.jsdelivr.net/npm/htmx-ext-ws@2.0.3/ws.js',
 'chunked-transfer': 'https://cdn.jsdelivr.net/npm/htmx-ext-transfer-encoding-chunked@0.4.0/transfer-encoding-chunked.js'}

In [ ]:
#| export
htmx_exts['ws4'] = 'https://unpkg.com/htmx.org@4.0.0-alpha7/dist/ext/hx-ws.js'

## WS4 patches

Two functions need patching:

1. **`_wrap_ws`** — htmx v2 sends headers as `HEADERS` (uppercase), v4 sends as `headers` (lowercase). The patch checks both keys.
2. **`_find_p`** — htmx v2 puts form fields at the top level of `data`, v4 nests them under `data['values']`. The patch checks both locations when `htmx4=True`.

No separate send function is needed — we use raw HTML for both v2 and v4.


In [ ]:
#| export
async def _find_p(conn, data, hdrs, arg:str, p:Parameter):
    "In `data` find param named `arg` of type in `p` (`arg` is ignored for body types)"
    anno = p.annotation
    # Special annotation types
    if isinstance(anno, type) and not isinstance(anno, GenericAlias):
        if issubclass(anno, HtmxHeaders): return _get_htmx(hdrs)
        if issubclass(anno, Starlette): return conn.scope['app']
        if issubclass(anno, HTTPConnection): return conn
        if issubclass(anno, State): return conn.scope['app'].state
        if anno is dict: return data
        if _is_body(anno):
            if 'session'.startswith(arg.lower()): return conn.scope.get('session', {})
            return await _from_body(conn, p, data)
    # Special param names with no annotations
    if anno is empty:
        if arg.lower()=='ws' or 'request'.startswith(arg.lower()): return conn
        if 'session'.startswith(arg.lower()): return conn.scope.get('session', {})
        if arg.lower()=='scope': return dict2obj(conn.scope)
        if arg.lower()=='data': return data
        if arg.lower()=='htmx': return _get_htmx(hdrs)
        if arg.lower()=='app': return conn.scope['app']
        if arg.lower()=='state': return conn.scope['app'].state
        if arg.lower()=='auth': return conn.scope.get('auth', None)
        if arg.lower()=='send':
            assert not isinstance(conn, Request), "`send` requires a websocket, not a `Request`"
            return partial(_send_ws, conn)
        if arg.lower()=='api': return ApiReturn(hdrs.get('accept')=='application/json')
        if arg.lower()=='body': return (await conn.body()).decode()
        if arg.lower() in ('hdrs','ftrs','bodykw','htmlkw'): return getattr(conn, arg.lower())
        if arg!='resp': warn(f"`{arg} has no type annotation and is not a recognised special name, so is ignored.")
        return None
    # Not a special name or a special annotation
    res = conn.path_params.get(arg, None)
    if res in (empty,None): res = conn.cookies.get(arg, None)
    if res in (empty,None): res = hdrs.get(snake2hyphens(arg), None)
    if res in (empty,None): res = conn.query_params.getlist(arg)
    if res==[]: res = None
    if res in (empty,None): res = data.get(arg, None)
    if res in (empty, None) and conn.scope['app'].htmx4: res = data.get('values', {}).get(arg, None) 
    if res in (empty,None):
        if p.default is empty:
            if isinstance(conn, Request): raise HTTPException(400, f"Missing required field: {arg}")
            raise ValueError(f"Missing required field: {arg}")
        res = p.default
    if anno is empty: return res
    try: return _fix_anno(anno, res)
    except ValueError as e:
        if isinstance(conn, Request): raise HTTPException(404, f"{conn.url.path}: {e}") from None
        raise

_core._find_p = _find_p

In [ ]:
#| export
async def _wrap_ws(ws, data, params):
    hdrs = {k.lower().replace('-','_'):v for k,v in (data.pop('HEADERS', {}) or data.pop('headers', {})).items()}
    return await _find_ps(ws, data, hdrs, params)

_core._wrap_ws = _wrap_ws

## WS Example

### WS Example with htmx2

```python
from asyncio import sleep

app = FastHTML(exts='ws')
rt = app.route

def mk_inp(): return Input(id='msg')
nid = 'notifications'

@rt('/')
async def get():
    cts = Div(
        Div(id=nid),
        Form(mk_inp(), id='form', ws_send=True),
        hx_ext='ws', ws_connect='/ws')
    return Titled('Websocket Test', cts)

async def on_connect(send): await send(Div('Hello, you have connected', id=nid))
async def on_disconnect( ): print('Disconnected!')

@app.ws('/ws', conn=on_connect, disconn=on_disconnect)
async def ws(msg:str, send):
    await send(Div('Hello ' + msg, id=nid))
    await sleep(2)
    return Div('Goodbye ' + msg, id=nid), mk_inp()

srv = JupyUvi(app)
```

### WS Example with htmx v4

```python
from asyncio import sleep

app = FastHTML(exts='ws', htmx=False, htmx4=True)
rt = app.route

def mk_inp(): return Input(id='msg')
nid = 'notifications'

@rt('/')
async def get():
    cts = Div(
        Div(id=nid),
        Form(mk_inp(), id='form', hx_ws_send=True),
        hx_ws_connect='/ws')
    return Titled('Websocket Test', cts)

async def on_connect(send): 
    await send(Div('Hello, you have connected', id=nid, hx_swap_oob=True))
async def on_disconnect(): print('Disconnected!')

@app.ws('/ws', conn=on_connect, disconn=on_disconnect)
async def ws(msg:str, send):
    await send(Div('Hello ' + msg, id=nid, hx_swap_oob=True))
    await sleep(2)
    # Send multiple elements in one message
    return (
        Div('Goodbye ' + msg, id=nid, hx_swap_oob=True),
        Input(id='msg', name='msg', value='', hx_swap_oob=True)
    )

srv = JupyUvi(app)
```

# SSE

## Migrating SSE from htmx v2 to v4

**Key Differences:**

**htmx v2:**
- Requires SSE extension: `Script(src="https://unpkg.com/htmx-ext-sse@2.2.1/sse.js")`
- Uses `hx_ext="sse"`, `sse_connect="/endpoint"`, `sse_swap="message"`
- Server uses `sse_message(data)` which includes `event: message\n`

**htmx v4:**
- SSE built into core htmx.js, no separate extension file needed
- Uses regular htmx attributes: `hx_get="/endpoint"`, `hx_trigger="load"`
- Server must omit `event:` line for normal swaps: `f"data: {to_xml(data)}\n\n"`

**Important:** In htmx v4, including `event:` triggers a custom DOM event instead of swapping content. For basic SSE swapping, use only `data:` lines.

**Updating multiple elements from one SSE stream (multiplexing):**

- **htmx v2:** Named events (`event: MyEvent`) + `sse_swap="MyEvent"` on each target element, different elements listen for different event names.
- **htmx v4:** Use `<hx-partial>` tags inside a single `data:` message. Each partial specifies its own `hx-target`:

```python
yield sse_message(
    (Partial(content_a, hx_target="#target-a"), Partial(content_b, hx_target="#target-b")),
    htmx4=True)
```

This replaces the named-event multiplexing pattern from v2.

## SSE patch

In [ ]:
#| export
def sse_message(elm, event='message', htmx4=False):
    "Convert element `elm` into a format suitable for SSE streaming"
    data = '\n'.join(f'data: {o}' for o in to_xml(elm).splitlines())
    # In htmx v4, omitting event: does default swap; including it triggers custom DOM event
    if htmx4 and event == 'message': return f'{data}\n\n'
    return f'event: {event}\n{data}\n\n'

import fasthtml.components
fasthtml.components.sse_message = sse_message

## SSE Example

### SSE Example with htmx v2

```python
import random
from asyncio import sleep

hdrs=(Script(src="https://unpkg.com/htmx-ext-sse@2.2.1/sse.js"),)
app,rt = fast_app(hdrs=hdrs)

@rt
def index():
    return Titled("SSE Random Number Generator",
        P("Generate random numbers, as the list grows scroll downwards."),
        Div(hx_ext="sse",
            sse_connect="/number-stream",
            hx_swap="beforeend show:bottom",
            sse_swap="message"))

shutdown_event = signal_shutdown()

async def number_generator():
    while not shutdown_event.is_set():
        data = Article(random.randint(1, 100))
        yield sse_message(data)
        await sleep(1)

@rt("/number-stream")
async def get(): return EventStream(number_generator())

srv = JupyUvi(app)
```

### SSE Example with htmx v4

```python
import random
from asyncio import sleep

app,rt = fast_app(htmx=False, htmx4=True)

@rt
def index():
    return Titled("SSE Random Number Generator",
        P("Generate random numbers, as the list grows scroll downwards."),
        Div(hx_get="/number-stream",
            hx_trigger="load",
            hx_swap="beforeend show:bottom"))

shutdown_event = signal_shutdown()

async def number_generator():
    while not shutdown_event.is_set():
        data = Article(random.randint(1, 100))
        yield sse_message(data, htmx4=True)
        await sleep(1)

@rt("/number-stream")
async def get(): return EventStream(number_generator())

srv = JupyUvi(app)
```

## SVG

## SVG OOB Swaps in htmx v4

In htmx v2, `SvgOob` wraps SVG content in a `<template>` tag, which was required for OOB swaps of SVG elements. In htmx v4, this `<template>` wrapper is no longer needed. SVG elements with `hx_swap_oob='true'` work directly.

**Migration:** Replace `SvgOob(...)` with plain `Svg(...)`

# Dev

**Important:** Do NOT re-import `from fasthtml.common import *` or `from fasthtml.jupyter import *` in this **DEV** section below when you copy code from example. Re-importing overwrites the patched functions (e.g. `def_hdrs`, `FastHTML.__init__`) with the originals and causes errors. All necessary symbols are already available from the earlier imports. And you also don't need to import `from hx4_patch.core import *`, we have already everything in this section


By the way, when I ask you to create a corresponding dialog, you should include `from fasthtml.common import *` and `from fasthtml.jupyter import *` before importing hx4_patch.

Search for me the list of project we haven't migrate to htmx4 yet and list me some of them. Migrated example with when you have `4` suffix at the end of the dialog (not .py)

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠



<details class='tool-usage-details'>
<summary>view(path='/app/data/fasthtml-example')</summary>

```json
{
  "id": "toolu_01Toz1NnhsweP5xdBumj7oJU",
  "call": {
    "function": "view",
    "arguments": {
      "path": "/app/data/fasthtml-example"
    }
  },
  "result": "<TRUNCATED>\u2026 contents of /app/data/fasthtml-example:\\n/app/data/fasthtml-example/02_chatbot \ud83d\udcc1\\n/app/data/fasthtml-example/todos2 \ud83d\udcc1\\n/app/data/fasthtml-example/devtools \ud83d\udcc1\\n/app/data/fasthtml-example/e_commerce \ud83d\udcc1\\n/app/data/fasthtml-example/todos2-hf \ud83d\udcc1\\n/app/data/fasthtml-example/chess_app \ud83d\udcc1\\n/app/data/fasthtml-example/LICENSE (11.1k)\\n/app/data/fasthtml-example/h2f \ud83d\udcc1\\n/app/data/fasthtml-example/image_app_simple \ud83d\udcc1\\n/app/data/fasthtml-example/code_editor \ud83d\udcc1\\n/app/data/fasthtml-example/cli_oauth_example \ud83d\udcc1\\n/app/data/fasthtml-example/CRAFT.ipynb (144.8k)\\n/app/data/fasthtml-example/README.md (2.2k)\\n/app/data/fasthtml-example/code_highlight_and_copy \ud83d\udcc1\\n/app/data/fasthtml-example/01_todo_app \ud83d\udcc1\\n/app/data/fasthtml-example/hacker_news_reskin \ud83d\udcc1\\n/app/data/fasthtml-example/file_upload_form_example \ud83d\udcc1\\n/app/data/fasthtml-example/htmx \ud83d\udcc1\\n/app/data/fasthtml-example/passkey \ud83d\udcc1\\n/app/data/fasthtml-example/shoelace.py (2.8k)\\n/app/data/fasthtml-example/00_game_of_life \ud83d\udcc1\\n/app/data/fasthtml-exampl\u2026</TRUNCATED>"
}
```

</details>



🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Here's what I found. **Already migrated** (have a `*4.ipynb` dialog):

| Folder | Migrated dialog |
|--------|----------------|
| `00_game_of_life` | `gol4.ipynb` |
| `01_todo_app` | `todo4.ipynb` |
| `02_chatbot` | `basic4.ipynb` |
| `04_sse` | `sse_rand_scroll4.ipynb`, `sse_clock_plus4.ipynb` |
| `echo` | `echo4.ipynb` |
| `helloworld` | `starter4.ipynb` |
| `infinite_scroll` | `infinite_scroll4.ipynb` |

**Not yet migrated** (some highlights):

| Folder | Source file(s) |
|--------|---------------|
| `todos1` | `main.py` |
| `todos2` | `main.py` |
| `e_commerce` | `main.py` |
| `hacker_news_reskin` | `main.py` |
| `doodle` | `main.py` |
| `image_app_simple` | `main.py` |
| `data_spot_check` | `main.py` |
| `annotate_text` | `main.py` |
| `code_editor` | `code_editor.py` |
| `chess_app` | `chess_app.py` |
| `03_pictionary` | `main.py` |
| `xtermjs` | `main.py` |
| `3d-component` | `main.py` |
| `htmx/` | multiple `.py` files (oob, ws, formdemo, etc.) |
| `04_sse` | `sse_chatbot.py`, `sse_rand.py`, `sse_clock.py`, `sse_clock_scroll.py` |
| `02_chatbot` | `ws_streaming.py`, `chunked_transfer.py`, `polling.py` (partially migrated — `polling4.py` exists but no `.ipynb`) |

Also `02_chatbot` has `ws4.py` and `polling4.py` as `.py` files but no corresponding `*4.ipynb` dialogs for those.

Which one would you like to tackle next?

Let's give me one from the htmx folder

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠



<details class='tool-usage-details'>
<summary>view(path='/app/data/fasthtml-example/htmx')</summary>

```json
{
  "id": "toolu_013dzz1dUfHjEgM1QeGUW9Sd",
  "call": {
    "function": "view",
    "arguments": {
      "path": "/app/data/fasthtml-example/htmx"
    }
  },
  "result": "<TRUNCATED>\u2026contents of /app/data/fasthtml-example/htmx:\n/app/data/fasthtml-example/htmx/favicon.ico (15.0k)\n/app/data/fasthtml-example/htmx/oob.py (0.9k)\n/app/data/fasthtml-example/htmx/counter-webc.py (0.7k)\n/app/data/fasthtml-example/htmx/ws_no.py (0.3k)\n/app/data/fasthtml-example/htmx/ws4.ipynb (1.6k)\n/app/data/fasthtml-example/htmx/surreal.py (0.8k)\n/app/data/fasthtml-example/htmx/formdemo.py (0.6k)\n/app/data/fasthtml-example/htmx/svgswap.py (0.9k)\n/app/data/fasthtml-example/htmx/ws.py (0.4k)\n/app/data/fasthtml-example/htmx/keys.py (0.5k)\n/app/data/fasthtml-example/htmx/counter-surreal.py (0.3k)\n/app/data/fasthtml-example/htmx/counter-uuid.py (0.4k)\n/app/data/fasthtml-example/htmx/formcb.py (0.4k)\n/app/data/fasthtml-example/htmx/counter-facet.py (0.6k)\n/app/data/fasthtml-example/htmx/utils.py (0.9k)\n/app/data/fasthtml-example/htmx/click-load.py (0.8k)\n/app/data/fasthtml-example/htmx/counter-rsjs.py (0.5k)\n/app/data/fasthtml-example/htmx/storage.py (0.4k)\n/app/data/fasthtml\u2026</TRUNCATED>"
}
```

</details>

Here are the htmx examples. Already migrated (`*4.ipynb`): **ws4**, **ws_no4**, **oob4**.

**Not yet migrated:**

| File | Description |
|------|-------------|
| `formdemo.py` | Form demo |
| `formcb.py` | Form callback |
| `click-load.py` | Click to load pattern |
| `surreal.py` | Surreal.js integration |
| `svgswap.py` | SVG swapping |
| `keys.py` | Keyboard shortcuts |
| `storage.py` | Storage |
| `counter-webc.py` | Counter (web component) |
| `counter-surreal.py` | Counter (surreal) |
| `counter-uuid.py` | Counter (UUID) |
| `counter-facet.py` | Counter (facet) |
| `counter-rsjs.py` | Counter (rsjs) |

Which one catches your eye? The simpler ones like `formdemo.py` or `click-load.py` might be good quick wins, while `oob.py` style ones exercise more htmx v4 features.

pull for me counter-surreal as it's in the code message below 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠



<details class='tool-usage-details'>
<summary>view(path='/app/data/fasthtml-example/htmx/counter-surreal.py')</summary>

```json
{
  "id": "toolu_01SzwqpUsxxJxX7isHA8XtSd",
  "call": {
    "function": "view",
    "arguments": {
      "path": "/app/data/fasthtml-example/htmx/counter-surreal.py"
    }
  },
  "result": "'from fasthtml.common import *\\n\\napp,rt = fast_app(live=True)\\n\\ndef Incrementer(start=0, btn_txt=\\'Increment\\'):\\n    return Section(\\n        Button(btn_txt),\\n        Prev(\"me(\\'output\\', m).value++;\"),\\n        Output(start))\\n\\n@rt\\ndef index():\\n    return Titled(\\'Surreal Incrementer\\',\\n        Incrementer(),\\n        Incrementer(5, \\'Do it\\'))\\n\\nserve()\\n'"
}
```

</details>

Here it is:

```python
from fasthtml.common import *

app,rt = fast_app(live=True)

def Incrementer(start=0, btn_txt='Increment'):
    return Section(
        Button(btn_txt),
        Prev("me('output', m).value++;"),
        Output(start))

@rt
def index():
    return Titled('Surreal Incrementer',
        Incrementer(),
        Incrementer(5, 'Do it'))

serve()
```

This one uses **surreal.js** (`Prev` + `me()`) for client-side incrementing — no htmx requests involved. It's purely a surreal.js demo, so the migration should be straightforward since there are no htmx attributes to update.

The main changes would be the standard boilerplate: `fast_app(htmx=False, htmx4=True)`, add the hx4_patch import, and swap `serve()` for `JupyUvi(app)`.

Want me to create the `counter-surreal4.ipynb` dialog for this?

In [ ]:
Section?

```python
def ft_hx[partial: 'section'](
    c:VAR_POSITIONAL, target_id:NoneType=None, hx_vals:NoneType=None, hx_target:NoneType=None, id:NoneType=None,
    cls:NoneType=None, title:NoneType=None, style:NoneType=None, accesskey:NoneType=None,
    contenteditable:NoneType=None, dir:NoneType=None, draggable:NoneType=None, enterkeyhint:NoneType=None,
    hidden:NoneType=None, inert:NoneType=None, inputmode:NoneType=None, lang:NoneType=None, popover:NoneType=None,
    spellcheck:NoneType=None, tabindex:NoneType=None, translate:NoneType=None, hx_get:NoneType=None,
    hx_post:NoneType=None, hx_put:NoneType=None, hx_delete:NoneType=None, hx_patch:NoneType=None,
    hx_trigger:NoneType=None, hx_swap:NoneType=None, hx_swap_oob:NoneType=None, hx_include:NoneType=None,
    hx_select:NoneType=None, hx_select_oob:NoneType=None, hx_indicator:NoneType=None, hx_push_url:NoneType=None,
    hx_confirm:NoneType=None, hx_disable:NoneType=None, hx_replace_url:NoneType=None, hx_disabled_elt:NoneType=None,
    hx_ext:NoneType=None, hx_headers:NoneType=None, hx_history:NoneType=None, hx_history_elt:NoneType=None,
    hx_inherit:NoneType=None, hx_params:NoneType=None, hx_preserve:NoneType=None, hx_prompt:NoneType=None,
    hx_request:NoneType=None, hx_sync:NoneType=None, hx_validate:NoneType=None, hx_on_blur:NoneType=None,
    hx_on_change:NoneType=None, hx_on_contextmenu:NoneType=None, hx_on_focus:NoneType=None,
    hx_on_input:NoneType=None, hx_on_invalid:NoneType=None, hx_on_reset:NoneType=None, hx_on_select:NoneType=None,
    hx_on_submit:NoneType=None, hx_on_keydown:NoneType=None, hx_on_keypress:NoneType=None, hx_on_keyup:NoneType=None,
    hx_on_click:NoneType=None, hx_on_dblclick:NoneType=None, hx_on_mousedown:NoneType=None,
    hx_on_mouseenter:NoneType=None, hx_on_mouseleave:NoneType=None, hx_on_mousemove:NoneType=None,
    hx_on_mouseout:NoneType=None, hx_on_mouseover:NoneType=None, hx_on_mouseup:NoneType=None,
    hx_on_wheel:NoneType=None, hx_on__abort:NoneType=None, hx_on__after_on_load:NoneType=None,
    hx_on__after_process_node:NoneType=None, hx_on__after_request:NoneType=None, hx_on__after_settle:NoneType=None,
    hx_on__after_swap:NoneType=None, hx_on__before_cleanup_element:NoneType=None,
    hx_on__before_on_load:NoneType=None, hx_on__before_process_node:NoneType=None,
    hx_on__before_request:NoneType=None, hx_on__before_swap:NoneType=None, hx_on__before_send:NoneType=None,
    hx_on__before_transition:NoneType=None, hx_on__config_request:NoneType=None, hx_on__confirm:NoneType=None,
    hx_on__history_cache_error:NoneType=None, hx_on__history_cache_miss:NoneType=None,
    hx_on__history_cache_miss_error:NoneType=None, hx_on__history_cache_miss_load:NoneType=None,
    hx_on__history_restore:NoneType=None, hx_on__before_history_save:NoneType=None, hx_on__load:NoneType=None,
    hx_on__no_sse_source_error:NoneType=None, hx_on__on_load_error:NoneType=None,
    hx_on__oob_after_swap:NoneType=None, hx_on__oob_before_swap:NoneType=None,
    hx_on__oob_error_no_target:NoneType=None, hx_on__prompt:NoneType=None, hx_on__pushed_into_history:NoneType=None,
    hx_on__replaced_in_history:NoneType=None, hx_on__response_error:NoneType=None, hx_on__send_abort:NoneType=None,
    hx_on__send_error:NoneType=None, hx_on__sse_error:NoneType=None, hx_on__sse_open:NoneType=None,
    hx_on__swap_error:NoneType=None, hx_on__target_error:NoneType=None, hx_on__timeout:NoneType=None,
    hx_on__validation_validate:NoneType=None, hx_on__validation_failed:NoneType=None,
    hx_on__validation_halted:NoneType=None, hx_on__xhr_abort:NoneType=None, hx_on__xhr_loadend:NoneType=None,
    hx_on__xhr_loadstart:NoneType=None, hx_on__xhr_progress:NoneType=None, kwargs:VAR_KEYWORD
):

```



```
partial(func, *args, **keywords) - new function with partial application
of the given arguments and keywords.
```



**File:** `/usr/local/lib/python3.12/functools.py`

**Type:** partial

What is a Section? why do we need it?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

`Section` is just FastHTML's way of creating an HTML `<section>` tag — a standard HTML5 semantic element for grouping related content.

In this example, it's important because of **surreal.js scoping**. The `Prev("me('output', m).value++")` uses `me()` to find a nearby `<output>` element. The `Section` wraps each incrementer's button + output together, so each one operates independently — clicking one button only increments *its own* counter, not the other one.

Reexplain me what is surreal and why do we need it?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠

**Surreal.js** is a tiny JavaScript library included by default in FastHTML. It provides a jQuery-like `me()` / `any()` API for selecting and manipulating DOM elements, with a key feature: **locality-scoped queries**.

Instead of searching the whole document, `me('output', m)` finds the `<output>` element *relative to the element that triggered the event* (`m` = the triggering element). This is what makes the two independent counters work without needing IDs or server round-trips.

**Why do we need it here?** Because this counter is **purely client-side** — no htmx request, no server involved. The button click just runs a tiny JS snippet inline via `Prev(...)`, which surreal executes in the context of the surrounding DOM.

`Prev(...)` is a FastHTML helper that generates an `hx-on:click` attribute (or `prev` in surreal terms) that runs the JS on the *previous sibling* — in this case the Button.

So the stack is:
- **HTML** — structure
- **Surreal.js** — local DOM manipulation
- **htmx** — not actually used here at all!

Does that make sense? Want me to go ahead and create the `counter-surreal4.ipynb` dialog?

Different between surreal and Lit?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

They serve very different purposes:

**Surreal.js** — tiny, DOM-manipulation focused
- Lightweight (~800 lines)
- `me()` / `any()` for locality-scoped DOM queries
- Designed to work *alongside* htmx for small client-side interactions
- No components, no reactivity system

**Lit** — a full web components framework by Google
- Builds reusable custom HTML elements (`<my-counter>`)
- Has a reactive property/state system (re-renders when data changes)
- Declarative templates with `html` tagged literals
- Much heavier, more structured

In short: Surreal is for **quick DOM tweaks** next to htmx, while Lit is for building **full encapsulated components** with their own state and lifecycle.

In this `counter-surreal.py` example, surreal was chosen precisely because the interaction is trivially simple — just increment a number. Lit would be overkill.

Let's have a simple HTML version without surreal then we will try to replace it with surreal

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great approach! Let's start simple.

A counter without surreal would use plain inline JavaScript. Something like:

```python
def Incrementer(start=0, btn_txt='Increment'):
    return Section(
        Output(start, id='count'),
        Button(btn_txt, onclick="document.getElementById('count').value++"))
```

Notice the difference — without surreal's `me()`, we need `document.getElementById(...)` and a hardcoded `id`. That means **two counters on the same page would conflict** unless you give each a unique id.

Try writing a full `index()` route and `serve()` / `JupyUvi` for this, and run it. Then we'll see the problem in action before fixing it with surreal!

Is it python?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes! It's Python — FastHTML lets you write HTML elements as Python functions. So `Section(...)`, `Button(...)`, `Output(...)` are all Python calls that generate the corresponding HTML tags.

For example, `Button("Increment", onclick="...")` generates:
```html
<button onclick="...">Increment</button>
```

Give the full example a try in a code cell — just write the `index()` route and a `JupyUvi(app)` at the bottom!

In [ ]:

app,rt = fast_app(live=True)
def Incrementer(start=0, btn_txt='Increment'):
    return Section(
        Button(btn_txt),
        Prev("me('output', m).value++;"),
        Output(start))

@rt
def index():
    return Titled('Surreal Incrementer',
        Incrementer(),
        Incrementer(5, 'Do it'))

srv = JupyUvi(app)

## Let's do the same thing without surreal

If we have 2 different counter, I guess we would need 2 different id right?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Exactly! With plain JavaScript you'd need something like `count1` and `count2` — and you'd have to manually manage those unique IDs.

Can you try writing that version with two counters, each with their own hardcoded id?

In [ ]:
app,rt = fast_app(live=True)
def Incrementer(start=0, btn_txt='Increment', id='count'):
    return Section(
        Button(btn_txt, onclick="alert('Click!')", id = id),
        Output(start))

@rt
def index():
    return Titled('Surreal Incrementer',
        Incrementer(),
        Incrementer(5, 'Do it'))

srv = JupyUvi(app)

ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): [errno 98] address already in use
